# Representation Stabilization — Experiment Inspection

Loads the raw CSVs from `results/`, calls the shared analysis functions,
and displays everything inline.  Run cells top-to-bottom.

| File | Produced by |
|---|---|
| `cka_results.csv` | `metrics/cka.py` |
| `drs_results.csv` | `metrics/drs.py` |
| `stabilization_*.csv` | `metrics/stabilization.py` |
| `surrogate_*.csv` | `surrogates/*.py` |

## 0 · Setup

In [ ]:
import os, sys, glob, logging
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, display

# Locate the repo root whether the kernel was started from notebooks/ or thesis/
_cwd = os.getcwd()
if os.path.exists(os.path.join(_cwd, 'analysis')):
    REPO_ROOT = _cwd
elif os.path.exists(os.path.join(_cwd, '..', 'analysis')):
    REPO_ROOT = os.path.abspath(os.path.join(_cwd, '..'))
else:
    raise RuntimeError(f'Cannot locate repo root from {_cwd}')

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

CONFIG_PATH = os.path.join(REPO_ROOT, 'configs', 'resnet18_cifar10.yaml')
RESULTS_DIR = os.path.join(REPO_ROOT, 'results')

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

TAU = float(config['tau'])
K   = int(config['K'])

# Guard against duplicate handlers when re-running setup cell
logger = logging.getLogger('notebook')
if not logger.handlers:
    logger.setLevel(logging.INFO)
    logger.addHandler(logging.StreamHandler(sys.stdout))

pd.set_option('display.float_format', '{:.6f}'.format)
pd.set_option('display.max_columns', None)

print(f'REPO_ROOT   : {REPO_ROOT}')
print(f'RESULTS_DIR : {RESULTS_DIR}')
print(f'tau = {TAU},  K = {K}')

## 1 · CKA — Raw Results

Each row is one consecutive checkpoint pair (every 5 epochs).  
`cka_change = 1 − CKA_value`.  The stabilisation criterion fires when `cka_change < tau`
for K consecutive pairs.

In [ ]:
cka_df = pd.read_csv(os.path.join(RESULTS_DIR, 'cka_results.csv'))
n_below = int(cka_df['below_tau'].sum())
print(f'{len(cka_df)} pairs  |  epochs {cka_df["epoch_curr"].min()}\u2013{cka_df["epoch_curr"].max()}')
print(f'Pairs below tau={TAU}: {n_below} / {len(cka_df)}')
print(f'CKA change range: {cka_df["cka_change"].min():.4f} \u2013 {cka_df["cka_change"].max():.4f}')
cka_df

### 1b · Stabilisation annotation

Light green: `cka_change < tau`.  Dark green + white text: the t* trigger row
(K-th consecutive below-tau pair).

In [ ]:
stab_cka_df = pd.read_csv(os.path.join(RESULTS_DIR, 'stabilization_cka.csv'))

def highlight_cka_streak(row):
    if row['is_t_star_trigger'] == 1:
        return ['background-color: #1a9641; color: white'] * len(row)
    if row['below_tau'] == 1:
        return ['background-color: #d4edda'] * len(row)
    return [''] * len(row)

stab_cka_df.style.apply(highlight_cka_streak, axis=1)

## 2 · DRS — Raw Results

DRS is computed every 10 epochs (expensive).  
`drs_change = 1 − DRS_value`.  
Unlike CKA, DRS measures *functional* similarity: the fraction of random 2D decision-region
slices where consecutive probes agree.

In [ ]:
drs_df = pd.read_csv(os.path.join(RESULTS_DIR, 'drs_results.csv'))
n_below_drs = int(drs_df['below_tau'].sum())
print(f'{len(drs_df)} pairs  |  epochs {drs_df["epoch_curr"].min()}\u2013{drs_df["epoch_curr"].max()}')
print(f'Pairs below tau={TAU}: {n_below_drs} / {len(drs_df)}')
print(f'DRS change range: {drs_df["drs_change"].min():.4f} \u2013 {drs_df["drs_change"].max():.4f}')
if n_below_drs == 0:
    print()
    print('DRS never crossed below tau — t* not detected for DRS.')
    print('The decision boundaries continue shifting even after CKA stabilises.')
    print('This is consistent with Davari et al. (2022): CKA stability does not imply functional stability.')
drs_df

### 2b · DRS stabilisation annotation

In [ ]:
stab_drs_df = pd.read_csv(os.path.join(RESULTS_DIR, 'stabilization_drs.csv'))

def highlight_drs_streak(row):
    if row['is_t_star_trigger'] == 1:
        return ['background-color: #1a9641; color: white'] * len(row)
    if row['below_tau'] == 1:
        return ['background-color: #d4edda'] * len(row)
    return [''] * len(row)

stab_drs_df.style.apply(highlight_drs_streak, axis=1)

## 3 · Stabilisation Summary

JOINT t* = max(t*_CKA, t*_DRS).  NOT_DETECTED in either metric propagates to JOINT.

In [ ]:
from analysis.plot_stability import load_stabilization_summary

summary    = load_stabilization_summary(RESULTS_DIR)
summary_df = pd.read_csv(os.path.join(RESULTS_DIR, 'stabilization_summary.csv'))

print('Parsed summary:', summary)
summary_df

## 4 · Surrogate Performance

Each surrogate was trained on penultimate-layer features frozen at the reported epoch.  
`accuracy_gap = surrogate_test_acc − network_test_acc`.  
A negative gap means the surrogate underperforms the full network head.

In [ ]:
surrogate_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'surrogate_*.csv')))
print(f'Found {len(surrogate_files)} surrogate result files:')
for p in surrogate_files:
    print(f'  {os.path.basename(p)}')

In [ ]:
surrogate_dfs = [pd.read_csv(p) for p in surrogate_files]
all_surrogates = pd.concat(surrogate_dfs, ignore_index=True)

# Columns present in every surrogate file
common_cols = ['epoch', 'surrogate_type', 'surrogate_train_acc',
               'surrogate_test_acc', 'network_test_acc', 'accuracy_gap']
available = [c for c in common_cols if c in all_surrogates.columns]

comparison = all_surrogates[available].copy()
comparison.style \
    .format({'surrogate_train_acc': '{:.4f}', 'surrogate_test_acc': '{:.4f}',
             'network_test_acc': '{:.4f}', 'accuracy_gap': '{:+.4f}'}) \
    .bar(subset=['accuracy_gap'], align='zero',
         color=['#d65f5f', '#5fba7d'])

## 5 · Stability Curves

Plots `1 − CKA` and `1 − DRS` over epochs.  
Green shading: checkpoint windows below tau.  Green dotted line: t* (CKA).  
`t* not detected` label: DRS never satisfied the criterion.

In [ ]:
from analysis.plot_stability import (
    load_cka_results,
    load_drs_results,
    plot_stability_curves,
)

cka_rows = load_cka_results(RESULTS_DIR)
drs_rows = load_drs_results(RESULTS_DIR)

stability_path = os.path.join(RESULTS_DIR, 'plot_stability.png')
plot_stability_curves(
    cka_rows=cka_rows,
    drs_rows=drs_rows,
    summary=summary,
    tau=TAU,
    K=K,
    output_path=stability_path,
    logger=logger,
)

Image(stability_path)

## 6 · CKA Heatmap

Requires `results/cka_pairwise_results.csv` (all N×N epoch pairs).  
If absent, generate it on the cluster:
```
python metrics/cka.py --mode pairwise --config configs/resnet18_cifar10.yaml
```

In [ ]:
pairwise_path = os.path.join(RESULTS_DIR, 'cka_pairwise_results.csv')

if not os.path.exists(pairwise_path):
    n_ckpts = len(cka_df) + 1
    n_pairs = n_ckpts * (n_ckpts + 1) // 2
    print('Pairwise CKA data not yet computed.')
    print(f'Estimated pairs: ~{n_pairs} for {n_ckpts} checkpoints.')
    print()
    print('Run from the repo root:')
    print('  python metrics/cka.py --mode pairwise --config configs/resnet18_cifar10.yaml')
else:
    from analysis.plot_cka_heatmap import (
        load_pairwise_cka,
        build_cka_matrix,
        plot_cka_heatmap,
    )
    pairwise_rows          = load_pairwise_cka(RESULTS_DIR)
    cka_matrix, epoch_list = build_cka_matrix(pairwise_rows)

    off_diag = cka_matrix.copy()
    np.fill_diagonal(off_diag, np.nan)
    print(f'Matrix: {cka_matrix.shape}  |  epochs {epoch_list[0]}\u2013{epoch_list[-1]}')
    print(f'Off-diagonal CKA range: {np.nanmin(off_diag):.4f} \u2013 {np.nanmax(off_diag):.4f}')

    heatmap_path = os.path.join(RESULTS_DIR, 'plot_cka_heatmap.png')
    plot_cka_heatmap(
        cka_matrix=cka_matrix,
        epoch_list=epoch_list,
        summary=summary,
        tau=TAU,
        K=K,
        output_path=heatmap_path,
        logger=logger,
    )
    display(Image(heatmap_path))

## 7 · Key Observations

*(Fill in after inspecting the plots above.)*

1. **CKA stabilisation epoch (t*)**: at what epoch does the curve first cross below tau and stay there?
2. **DRS divergence**: DRS change remains well above tau.  Does this support or challenge the stabilisation story?  See Davari et al. (2022).
3. **Surrogate sufficiency**: do the surrogates match the full network accuracy at the freeze epoch?  What is the accuracy gap?
4. **Phase structure** *(heatmap)*: are there visible block structures corresponding to the three training phases from Sharon & Dar (2024)?
5. **Implications**: if CKA stabilises but DRS does not, what does that say about the relationship between geometric and functional stability?